# Retail Price Intelligence — Full Analysis

This notebook walks through the complete price intelligence workflow:

1. **Data Generation** — create realistic synthetic retail pricing data
2. **Exploratory Data Analysis** — understand price distributions, trends, and patterns
3. **Anomaly Detection** — identify pricing errors and unusual movements
4. **Competitive Analysis** — price positioning, leadership, and gap analysis
5. **Business Insights** — actionable takeaways for trade marketing

---

**Author:** Eduardo Moraga | [eduardomoraga.github.io](https://eduardomoraga.github.io)  
**Context:** 20 electronics products × 5 retailers × 365 days of daily pricing in Chilean Pesos (CLP)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)

TEMPLATE = 'plotly_dark'
COLORS = ['#00d4aa', '#ff6b6b', '#ffd93d', '#6bcbff', '#c084fc', '#ff9f43']

## 1. Data Generation & ETL Pipeline

In [ ]:
from src.data_generator import generate_retail_data
from src.etl_pipeline import transform

# Generate 365 days of synthetic retail pricing data
raw = generate_retail_data(n_days=365, start_date='2024-01-01', seed=42)
print(f'Raw data: {raw.shape[0]:,} rows × {raw.shape[1]} columns')
raw.head()

In [ ]:
# Run the transform step to enrich the data
df = transform(raw)
print(f'Enriched data: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
df.info()

## 2. Exploratory Data Analysis

### 2.1 Price Distributions by Brand

In [ ]:
fig = px.box(
    df, x='brand', y='price', color='brand',
    title='Price Distribution by Brand (CLP)',
    template=TEMPLATE,
    color_discrete_sequence=COLORS,
)
fig.update_layout(showlegend=False, height=500)
fig.show()

### 2.2 Price Trends Over Time

In [ ]:
# Average daily price by brand
daily_brand = df.groupby(['date', 'brand'])['price'].mean().reset_index()

fig = px.line(
    daily_brand, x='date', y='price', color='brand',
    title='Average Daily Price by Brand',
    template=TEMPLATE,
    color_discrete_sequence=COLORS,
    labels={'price': 'Avg Price (CLP)'},
)
fig.update_layout(height=500)
fig.show()

### 2.3 Price by Retailer

In [ ]:
fig = px.box(
    df, x='retailer', y='price_vs_market_pct', color='retailer',
    title='Price vs Market Average by Retailer',
    template=TEMPLATE,
    color_discrete_sequence=COLORS,
    labels={'price_vs_market_pct': '% vs Market Avg'},
)
fig.update_layout(showlegend=False, height=450)
fig.show()

### 2.4 Discount Analysis

In [ ]:
promo = df[df['is_promoted'] == True]

fig = px.histogram(
    promo, x='discount_pct', color='retailer',
    title='Discount Depth Distribution During Promotions',
    template=TEMPLATE,
    color_discrete_sequence=COLORS,
    barmode='overlay',
    opacity=0.7,
    nbins=30,
    labels={'discount_pct': 'Discount %'},
)
fig.update_layout(height=450)
fig.show()

### 2.5 Stock Availability

In [ ]:
stock_rate = (
    df.groupby(['retailer', 'brand'])['in_stock']
    .mean()
    .reset_index()
    .rename(columns={'in_stock': 'availability_rate'})
)

fig = px.bar(
    stock_rate, x='retailer', y='availability_rate', color='brand',
    barmode='group',
    title='Stock Availability Rate by Retailer and Brand',
    template=TEMPLATE,
    color_discrete_sequence=COLORS,
    labels={'availability_rate': 'Availability Rate'},
)
fig.update_layout(height=450)
fig.show()

## 3. Anomaly Detection

We use a 4-method ensemble:
- Z-Score (rolling 30-day window)
- IQR (rolling 30-day window)
- Isolation Forest (sklearn)
- Rate of Change (>15% single-day drop)

An observation is flagged when **2+ methods agree**.

In [ ]:
from src.anomaly_detection import detect_anomalies, anomaly_summary

alerts = detect_anomalies(df)
print(f'Total ensemble anomalies: {len(alerts)}')
alerts.head(10)

In [ ]:
# Anomaly breakdown
summary = anomaly_summary(alerts)
print('--- By Severity & Type ---')
display(summary['by_type'])
print('\n--- By Retailer ---')
display(summary['by_retailer'])
print('\n--- By Brand ---')
display(summary['by_brand'])

In [ ]:
# Visualize anomalies on a sample product
sample_product = 'Samsung Crystal UHD 55"'
sample = df[df['product'] == sample_product].sort_values('date')
sample_alerts = alerts[alerts['product'] == sample_product] if not alerts.empty else pd.DataFrame()

fig = px.line(
    sample, x='date', y='price', color='retailer',
    title=f'Price Trend with Anomalies: {sample_product}',
    template=TEMPLATE,
    color_discrete_sequence=COLORS,
)

if not sample_alerts.empty:
    fig.add_trace(go.Scatter(
        x=sample_alerts['date'], y=sample_alerts['price'],
        mode='markers',
        marker=dict(color='red', size=14, symbol='x'),
        name='Anomaly',
    ))

fig.update_layout(height=500)
fig.show()

## 4. Competitive Analysis

### 4.1 Price Leadership

In [ ]:
from src.price_analysis import (
    price_leader_analysis, price_gap_matrix,
    price_vs_market_by_retailer, brand_price_index,
    promotional_effectiveness,
)

leaders = price_leader_analysis(df)

fig = px.bar(
    leaders, x='brand', y='pct_cheapest', color='retailer',
    barmode='group',
    title='Price Leadership: % of Days as Cheapest Retailer',
    template=TEMPLATE,
    color_discrete_sequence=COLORS,
    labels={'pct_cheapest': '% Days Cheapest'},
)
fig.update_layout(height=450)
fig.show()

### 4.2 Retailer Price Gap Matrix

In [ ]:
gap = price_gap_matrix(df)

fig = px.imshow(
    gap.values.astype(float),
    x=gap.columns.tolist(),
    y=gap.index.tolist(),
    color_continuous_scale='RdYlGn_r',
    title='Retailer vs Retailer Avg Price Gap (positive = row is more expensive)',
    template=TEMPLATE,
    aspect='auto',
)
fig.update_layout(height=450)
fig.show()

### 4.3 Price vs Market Heatmap

In [ ]:
pvm = price_vs_market_by_retailer(df)
heat = pvm.pivot_table(index='product', columns='retailer', values='avg_price_vs_market')

fig = px.imshow(
    heat.values, x=heat.columns.tolist(), y=heat.index.tolist(),
    color_continuous_scale=['#00d4aa', '#1a2332', '#ff4444'],
    title='Avg Price vs Market by Product × Retailer',
    template=TEMPLATE,
    aspect='auto',
)
fig.update_layout(height=600)
fig.show()

### 4.4 Brand Price Index Over Time

In [ ]:
bpi = brand_price_index(df)

fig = px.line(
    bpi, x='date', y='avg_price_index', color='brand',
    title='Brand Price Index (Base 100 = Day 1)',
    template=TEMPLATE,
    color_discrete_sequence=COLORS,
)
fig.update_layout(height=450)
fig.show()

### 4.5 Promotional Effectiveness

In [ ]:
promo_eff = promotional_effectiveness(df)

print('=== Promotional Summary by Retailer ===')
display(promo_eff['summary'])

print('\n=== Price Recovery After Promotions ===')
display(promo_eff['recovery'])

In [ ]:
fig = px.bar(
    promo_eff['summary'], x='retailer', y='avg_discount',
    title='Average Promotional Discount by Retailer',
    template=TEMPLATE,
    color_discrete_sequence=['#00d4aa'],
    labels={'avg_discount': 'Avg Discount'},
)
fig.update_layout(height=400)
fig.show()

## 5. Business Insights

Auto-generated, actionable insights for trade marketing decision-making.

In [ ]:
from src.report_generator import generate_insights

insights = generate_insights(df, alerts)

for i, insight in enumerate(insights, 1):
    sev_icon = {'high': '\U0001f534', 'medium': '\U0001f7e1', 'low': '\U0001f535'}.get(insight['severity'], '')
    print(f"{i:2d}. {sev_icon} [{insight['category'].upper()}] {insight['text']}")

---

## Key Takeaways

1. **Amazon and MercadoLibre consistently undercut** department stores (Falabella, Paris, Ripley) by 4-6% on average — a pattern that mirrors real LATAM marketplace dynamics.

2. **Promotional events create measurable price drops** of 15-28%, with Cyber Day and Black Friday being the deepest. Price recovery after promotions takes 3-7 days on average.

3. **The 4-method ensemble anomaly detector** effectively identifies both deliberate promotional drops and genuine pricing errors, with the 2-vote threshold significantly reducing false positives.

4. **Apple products show the lowest price volatility** (tight MAP enforcement in real life), while Xiaomi shows the highest (aggressive marketplace competition).

5. **Stock availability correlates weakly with price changes** — suggesting that in electronics retail, availability is more supply-driven than demand-driven.

---

*Built by Eduardo Moraga — Trade Marketing × Data Science*